In [ ]:
%%capture
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

# AQUÍ ES DONDE SE DEFINE 'gc'
gc = gspread.authorize(creds)
#  CONEXION AL DRIVE Y HABILITAR PDF

!pip install fpdf2

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd
import numpy as np
import os
import datetime
from fpdf import FPDF
from fpdf.enums import XPos, YPos
from IPython.display import display

print("📥 Leyendo datos frescos de Google Sheets...")
ID_SHEET = 'https://docs.google.com/spreadsheets/d/1f0i3my1rGrLs6RL5oJhxSIjBIMGYRUI3APiJa1_m8Yk/edit?gid=0#gid=0'
print("🔍 Conectando directamente al ID del archivo...")
spreadsheet = gc.open_by_url(ID_SHEET)
sheet_aportantes = spreadsheet.worksheet('M_APORTANTES')
sheet_empleados = spreadsheet.worksheet('M_EMPLEADOS')
sheet_novedades = spreadsheet.worksheet('T_NOVEDADES')

# --- 1. CARGA Y LIMPIEZA INICIAL ---
data_apo = sheet_aportantes.get_all_values()
data_emp = sheet_empleados.get_all_values()
data_nov = sheet_novedades.get_all_values()
df_apo = pd.DataFrame(data_apo[1:], columns=data_apo[0])
df_emp = pd.DataFrame(data_emp[1:], columns=data_emp[0])
df_nov = pd.DataFrame(data_nov[1:], columns=data_nov[0])

# Limpieza de columnas y normalización
for df in [df_nov, df_emp, df_apo]:
    df.columns = df.columns.astype(str).str.strip().str.upper().str.replace(r'\s+', '_', regex=True)

# -------------------------------------------------------------------------
# ✅ FILTRO DE NÓMINAS A GENERAR
# -------------------------------------------------------------------------
if 'GENERAR_NOMINA' in df_nov.columns:
    filtro = df_nov['GENERAR_NOMINA'].astype(str).str.strip().str.upper().isin(['SI', 'SÍ', 'TRUE', '1', 'VERDADERO'])
    df_nov = df_nov[filtro]
    print(f"✅ Filtro aplicado: Se procesarán {len(df_nov)} nóminas según la columna GENERAR_NOMINA.")
else:
    print("⚠️ No se encontró la columna 'GENERAR_NOMINA' en T_NOVEDADES. Se procesarán todos los registros.")
# -------------------------------------------------------------------------

df_apo['ID_APORTANTE'] = df_apo['ID_APORTANTE'].astype(str).str.strip()
df_emp['ID_APORTANTE'] = df_emp['ID_APORTANTE'].astype(str).str.strip()
df_emp['ID_CONTRATO'] = df_emp['ID_CONTRATO'].astype(str).str.strip()
df_nov['ID_CONTRATO'] = df_nov['ID_CONTRATO'].astype(str).str.strip()

# --- 2. UNIÓN DE TABLAS ---
df_temp = pd.merge(df_nov, df_emp, on='ID_CONTRATO', how='left')
df_final = pd.merge(df_temp, df_apo, on='ID_APORTANTE', how='left')

# =====================================================================
# PARTE 1: CÁLCULO DE NÓMINA (Ajustado con Bono Diario y Préstamos)
# =====================================================================

# --- 3. PARÁMETROS 2026 ---
print("🧮 Calculando nómina...")
SMLV_2026 = 1750905
SMLD_2026 = SMLV_2026 / 30
PISO_TP_BONO = SMLD_2026 * (7/6)
AUX_TTE_MES = 249095
LIMITE_AUX = SMLV_2026 * 2
HR_MES = 220
PORCENTAJE_LEY = 0.04

FACTORES = {'HED': 1.25, 'HEN': 1.75, 'HEDF': 2.05, 'HENF': 2.55,'RN': 0.35, 'RDN': 0.80, 'RNF': 1.15}

df_final.columns = df_final.columns.str.strip().str.upper()

# --- 4. NORMALIZACIÓN NUMÉRICA (Se añade PRIMA_CALC) ---
cols_limpiar = ['SALARIO_BASE', 'VLR_BONO', 'PRESTAMOS', 'SAL_ESPECIE', 'PRIMA_CALC'] # <-- Se añade PRIMA_CALC aquí
for col in cols_limpiar:
    if col in df_final.columns:
        df_final[col] = df_final[col].astype(str).str.replace(r'[\$,.]', '', regex=True)

# Asegurar que existan columnas clave para evitar errores antes de operar
if 'HORAS_LABORADAS' not in df_final.columns:
    df_final['HORAS_LABORADAS'] = 0
if 'PRESTAMOS' not in df_final.columns:
    df_final['PRESTAMOS'] = 0
if 'PRIMA_CALC' not in df_final.columns:
    df_final['PRIMA_CALC'] = 0

cols_num = cols_limpiar + ['DIAS_LABORADOS', 'HORAS_LABORADAS', 'DIAS_VACACIONES', 'DIAS_INCAPACIDAD'] + list(FACTORES.keys())
cols_existentes = [col for col in cols_num if col in df_final.columns]
df_final[cols_existentes] = df_final[cols_existentes].apply(pd.to_numeric, errors='coerce').fillna(0)

# --- 5. & 6. LIQUIDACIÓN VECTORIZADA DE NÓMINA ---

# A. Booleanos, Variables Base y ESTADO
es_smlv = df_final.get('ES_SMLV', '').astype(str).str.strip().str.upper().isin(['VERDADERO', 'TRUE', 'SI', '1'])
con_bono = df_final.get('CON_BONO', '').astype(str).str.strip().str.upper().isin(['VERDADERO', 'TRUE', 'SI', '1'])
tiene_aux = df_final.get('TIENE_AUX', '').astype(str).str.strip().str.upper().isin(['VERDADERO', 'TRUE', 'SI', 'SÍ', '1'])
tipo_contrato = df_final.get('TIPO_CONTRATO', 'TIEMPO COMPLETO').astype(str).str.strip().str.upper()
estado_empleado = df_final.get('ESTADO_EMPLEADO', 'ACTIVO').astype(str).str.strip().str.upper()
periodo_pago = df_final.get('PERIODO_PAGO', 'QUINCENAL').astype(str).str.strip().str.upper()

# Distribución de Días
d_vac = df_final.get('DIAS_VACACIONES', 0)
d_inc = df_final.get('DIAS_INCAPACIDAD', 0)
d_lab_total = np.where(df_final['HORAS_LABORADAS'] > 0, df_final['HORAS_LABORADAS'] / 8, df_final['DIAS_LABORADOS'])

# Días efectivamente trabajados
dias_efectivos_trabajo = np.maximum(d_lab_total - d_vac - d_inc, 0)

sal_base_raw = df_final.get('SALARIO_BASE', 0)
sal_especie_raw = df_final.get('SAL_ESPECIE', 0)

# CONDICIÓN: Empleado Interno
df_final['TOTAL_BASE_MENSUAL'] = np.where(
            tipo_contrato == "EMPLEADO INTERNO",
            sal_base_raw + sal_especie_raw,
            sal_base_raw
        )

sal_base_input = df_final['TOTAL_BASE_MENSUAL']

val_diario_propuesto = np.where(tipo_contrato == "TIEMPO PARCIAL", sal_base_input, sal_base_input / 30)

val_diario_validado = np.where(
                              (tipo_contrato == "TIEMPO PARCIAL") & (con_bono),
                              np.maximum(val_diario_propuesto, PISO_TP_BONO),
                              val_diario_propuesto)

sal_base_mensual_equiv = val_diario_validado * 30

# B. Estructura Salarial
df_final['SAL_REF'] = np.where(es_smlv, SMLV_2026, sal_base_mensual_equiv)
df_final['BONO_REF'] = np.where(con_bono, df_final['VLR_BONO'], 0)

# C. Devengados
valor_dia_total = df_final['SAL_REF'] / 30

df_final['VAL_DIA_ESPECIE'] = np.where(tipo_contrato == "EMPLEADO INTERNO", sal_especie_raw / 30, 0)
df_final['VAL_DIA_EFECTIVO'] = valor_dia_total - df_final['VAL_DIA_ESPECIE']

df_final['SUELDO_EFECTIVO_PAGADO'] = df_final['VAL_DIA_EFECTIVO'] * dias_efectivos_trabajo
df_final['SAL_ESPECIE_PAGADO'] = df_final['VAL_DIA_ESPECIE'] * dias_efectivos_trabajo
df_final['SUELDO_TRABAJADO'] = df_final['SUELDO_EFECTIVO_PAGADO'] + df_final['SAL_ESPECIE_PAGADO']

# 2. Pago Vacaciones e Incapacidades
df_final['PAGO_VACACIONES'] = valor_dia_total * d_vac
pago_inc_diario = np.maximum(valor_dia_total * 0.6667, SMLD_2026)
df_final['PAGO_INCAPACIDAD'] = pago_inc_diario * d_inc

# 4. Bonos y Extras
df_final['BONO_PAGADO'] = np.where(
    tipo_contrato == "TIEMPO PARCIAL",
    df_final['BONO_REF'] * d_lab_total,
    (df_final['BONO_REF'] / 30) * d_lab_total
)
valor_hora = df_final['SAL_REF'] / HR_MES
df_final['TOTAL_EXTRAS'] = 0
for cod, factor in FACTORES.items():
    if cod in df_final.columns:
        df_final['TOTAL_EXTRAS'] += df_final[cod] * valor_hora * factor

# 5. Auxilio de Transporte
bono_mensual_tope = np.where(tipo_contrato == "TIEMPO PARCIAL", df_final['BONO_REF'] * 30, df_final['BONO_REF'])
cond_aux = tiene_aux & ((df_final['SAL_REF'] + bono_mensual_tope) <= LIMITE_AUX)
df_final['VAL_AUX_TTE'] = np.where(cond_aux, (AUX_TTE_MES / 30) * dias_efectivos_trabajo, 0)

# E. IBC PILA (Considerando Vacaciones e Incapacidades)
ibc_tiempo_completo = np.maximum(df_final['SUELDO_TRABAJADO'] + df_final['PAGO_VACACIONES'] + df_final['PAGO_INCAPACIDAD'] + df_final['TOTAL_EXTRAS'], (SMLV_2026 / 30) * d_lab_total)

dias_proyectados = np.where(periodo_pago == 'QUINCENAL', d_lab_total * 2, d_lab_total)
cond_parcial = [dias_proyectados <= 7, dias_proyectados <= 14, dias_proyectados <= 21, dias_proyectados > 21]
opciones_parcial_mes = [SMLV_2026 * 0.25, SMLV_2026 * 0.50, SMLV_2026 * 0.75, SMLV_2026]

ibc_parcial_proporcional = np.where(
    periodo_pago == 'QUINCENAL',
    np.select(cond_parcial, opciones_parcial_mes) / 2,
    np.select(cond_parcial, opciones_parcial_mes)
)

df_final['IBC_PILA'] = np.where(tipo_contrato == "TIEMPO PARCIAL", ibc_parcial_proporcional + df_final['TOTAL_EXTRAS'], ibc_tiempo_completo)

# E. Deducciones
def redondear_centena_superior_vec(serie):
    return np.ceil(np.maximum(serie, 0) / 100) * 100

eps_exento = df_final.get('EPS', '').astype(str).str.strip().str.upper().str.contains('N/A', na=False)
afp_exento = df_final.get('AFP', '').astype(str).str.strip().str.upper().str.contains('N/A', na=False)

df_final['SALUD_4'] = np.where(eps_exento, 0, redondear_centena_superior_vec(df_final['IBC_PILA'] * PORCENTAJE_LEY))
df_final['PENSION_4'] = np.where(afp_exento, 0, redondear_centena_superior_vec(df_final['IBC_PILA'] * PORCENTAJE_LEY))

# F. Totales Finales (Modificado para incluir PRIMA_CALC)
df_final['SUELDO_PAGADO'] = df_final['SUELDO_TRABAJADO'] + df_final['PAGO_VACACIONES'] + df_final['PAGO_INCAPACIDAD']

# Se suma la columna PRIMA_CALC al total devengado
df_final['TOTAL_DEVENGADO'] = (df_final['SUELDO_PAGADO'] +
                               df_final['BONO_PAGADO'] +
                               df_final['TOTAL_EXTRAS'] +
                               df_final['VAL_AUX_TTE'] +
                               df_final['PRIMA_CALC']) # <-- Adición de la prima al devengado

df_final['TOTAL_DEDUCIDO'] = df_final['SALUD_4'] + df_final['PENSION_4'] + df_final['PRESTAMOS'] + df_final['SAL_ESPECIE_PAGADO']
df_final['NETO_PAGAR'] = df_final['TOTAL_DEVENGADO'] - df_final['TOTAL_DEDUCIDO']

# --- 7. VALIDACIÓN DE ESTADO DEL EMPLEADO (Se añade PRIMA_CALC) ---
es_retirado = estado_empleado == 'RETIRADO'

cols_a_ceros = [
    'SUELDO_PAGADO', 'SUELDO_EFECTIVO_PAGADO', 'SAL_ESPECIE_PAGADO',
    'BONO_PAGADO', 'TOTAL_EXTRAS', 'VAL_AUX_TTE', 'PRESTAMOS', 'PRIMA_CALC', # <-- Se añade PRIMA_CALC aquí
    'IBC_PILA', 'SALUD_4', 'PENSION_4', 'TOTAL_DEVENGADO', 'TOTAL_DEDUCIDO', 'NETO_PAGAR'
]
for col in cols_a_ceros:
    df_final[col] = np.where(es_retirado, 0, df_final[col])

# --- 8. VISUALIZACIÓN ---
pd.options.display.float_format = '{:,.0f}'.format
print("✅ NÓMINA: PROPORCIONALIDAD CON SOPORTE PARA HORAS/DÍAS SALTEADOS (INCLUYE PRIMA)")
cols_finales = [
    'NOMBRE_EMPLEADO', 'DIAS_VACACIONES', 'DIAS_INCAPACIDAD','TIPO_CONTRATO', 'PRESTAMOS', 'PRIMA_CALC', 'IBC_PILA', 'VAL_AUX_TTE',
    'BONO_PAGADO', 'TOTAL_DEVENGADO', 'TOTAL_DEDUCIDO', 'HORAS_LABORADAS', 'DIAS_LABORADOS', 'NETO_PAGAR'
]
cols_mostrar = [c for c in cols_finales if c in df_final.columns]
display(df_final[cols_mostrar].head(15))

📥 Leyendo datos frescos de Google Sheets...
🔍 Conectando directamente al ID del archivo...
✅ Filtro aplicado: Se procesarán 274 nóminas según la columna GENERAR_NOMINA.
🧮 Calculando nómina...
✅ NÓMINA: PROPORCIONALIDAD CON SOPORTE PARA HORAS/DÍAS SALTEADOS (INCLUYE PRIMA)


,NOMBRE_EMPLEADO,DIAS_VACACIONES,DIAS_INCAPACIDAD,TIPO_CONTRATO,PRESTAMOS,PRIMA_CALC,IBC_PILA,VAL_AUX_TTE,BONO_PAGADO,TOTAL_DEVENGADO,TOTAL_DEDUCIDO,HORAS_LABORADAS,DIAS_LABORADOS,NETO_PAGAR
0,MARIBEL QUINTERO QUINTERO,0,0,Tiempo Completo,0,422222,"875,452","124,547","52,000","1,474,222","70,200",0,15,"1,404,022"
1,YEIMY JIUMARA ESPITIA MALAVER,0,0,Tiempo Completo,0,1000000,"875,452","124,547",0,"2,000,000","35,100",0,15,"1,964,900"
2,ANGIE TATIANA CAICEDO CRUZ,0,0,Tiempo Parcial,0,58333,"437,726","49,819",0,"516,696","17,600",0,6,"499,096"
3,KAROLD VIVIANA JIMENEZ,0,0,Tiempo Completo,0,1000000,"875,452","124,547",0,"2,000,000","70,200",0,15,"1,929,800"
4,MARIA LUZ LÓPEZ,0,0,Tiempo Completo,0,1013821,"1,387,990","124,547",0,"2,526,359","55,600",0,15,"2,470,759"
5,YHOJANIS RODRIGUEZ BELLO,0,0,Tiempo Parcial,0,415102,"437,726","58,122",0,"1,080,103","17,600",0,7,"1,062,503"
6,SANDRA MARCELA PARRA BARRIGA,0,0,Tiempo Parcial,0,357604,"437,726","33,213",0,"717,605","17,600",0,4,"700,005"
7,YURANI ESTEFANI RUIZ DIAZ,0,0,Tiempo Completo,0,968333,"925,452","124,547",0,"2,018,333","74,200",0,15,"1,944,133"
8,NIDIA RAQUEL TORRES URQUIJO,0,0,Tiempo Completo,0,1000000,"875,452","124,547",0,"2,000,000","70,200",0,15,"1,929,800"
9,ROSIRA ISABEL RUIZ MARTÍNEZ,0,0,Tiempo Parcial,0,496672,"437,726","49,819",0,"1,096,673","17,600",0,6,"1,079,073"


In [ ]:
# =====================================================================
# PARTE 2: GENERACIÓN DE PDF COMPROBANTES (Ajustado con Prima)
# =====================================================================

logo_path = '/content/drive/MyDrive/LOGO_UFK.jpg'
ruta_drive_comprobantes = '/content/drive/MyDrive/Nomina_2026/comprobantes'

# Nombres amigables para el detalle de extras
NOMBRES_EXTRAS = {
    'HED': 'HR. EXTRA DIURNA (1.25)',
    'HEN': 'HR. EXTRA NOCTURNA (1.75)',
    'HEDF': 'HR. EXTRA DOM/FEST DIURNA (2.05)',
    'HENF': 'HR. EXTRA DOM/FEST NOCTURNA (2.55)',
    'RN': 'RECARGO NOCTURNO (0.35)',
    'RDN': 'RECARGO DOM/FEST DIURNO (0.80)',
    'RNF': 'RECARGO FESTIVO NOCTURNO (1.15)'
}

def forzar_numero(valor):
    """Convierte cualquier dato de Pandas a un float único de Python"""
    try:
        if isinstance(valor, pd.Series):
            valor = valor.iloc[0] # Si hay duplicados, toma el primero
        return float(valor) if pd.notnull(valor) else 0.0
    except:
        return 0.0

def formatear_periodo(valor):
    """Convierte fechas a texto en MAYÚSCULAS"""
    if pd.isnull(valor): return "SIN PERIODO"

    if isinstance(valor, (pd.Timestamp, datetime.datetime)):
        return valor.strftime('%B %Y').upper()
    try:
        return pd.to_datetime(valor).strftime('%B %Y').upper()
    except:
        return str(valor).upper()

class PDF(FPDF):
    def __init__(self, datos_empleador, periodo_liq):
        super().__init__()
        self.emp_nombre = datos_empleador['nombre']
        self.emp_nit = datos_empleador['nit']
        self.emp_tipo = datos_empleador['tipo']
        self.periodo_liq = periodo_liq

    def header(self):
        # 1. MARCA DE AGUA
        if logo_path and os.path.exists(logo_path):
            with self.local_context(fill_opacity=0.2):
                self.image(logo_path, x=10, y=10, w=190)

        # 2. DATOS DEL EMPLEADOR
        self.set_font('helvetica', 'B', 11)
        self.cell(0, 5, self.emp_nombre, align='R', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        self.set_font('helvetica', '', 9)
        self.cell(0, 4, f"RUT: {self.emp_nit}", align='R', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        self.cell(0, 4, f"Tipo: {self.emp_tipo}", align='R', new_x=XPos.LMARGIN, new_y=YPos.NEXT)

        self.ln(4)

        # 3. TÍTULO Y PERIODO DINÁMICO
        self.set_font('helvetica', 'B', 12)
        self.cell(0, 10, 'COMPROBANTE INDIVIDUAL DE PAGO DE NÓMINA', align='C', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        self.set_font('helvetica', '', 11)
        self.cell(0, 5, f'Periodo de Pago: {self.periodo_liq}', align='C', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        self.ln(5)

def exportar_nomina_multicliente(df):
    ruta_raiz_clientes = '/content/drive/MyDrive/CLIENTES'

    if not os.path.exists(ruta_drive_comprobantes):
        os.makedirs(ruta_drive_comprobantes)

    df.columns = df.columns.str.strip().str.upper()
    df_activos = df[df['ESTADO_EMPLEADO'] != 'RETIRADO'].copy()

    factores_dict = {'HED': 1.25, 'HEN': 1.75, 'HEDF': 2.05, 'HENF': 2.55, 'RN': 0.35, 'RDN': 0.80, 'RNF': 1.15}

    for _, row in df_activos.iterrows():
        # --- 1. PROCESAR DATOS BÁSICOS ---
        periodo_actual = formatear_periodo(row.get('PERIODO_LIQ'))
        id_contrato = row.get('ID_CONTRATO', 'SIN_ID')
        if isinstance(id_contrato, pd.Series):
            id_contrato = id_contrato.iloc[0]
        id_contrato = str(id_contrato).strip()

        id_cliente = str(row.get('ID_APORTANTE', 'SIN_CLIENTE')).strip()
        id_empleado = str(row.get('ID_EMPLEADO', 'SIN_EMPLEADO')).strip()

        # --- 2. CONFIGURAR RUTA DINÁMICA ---
        ruta_personalizada = os.path.join(ruta_raiz_clientes, id_cliente, id_empleado, 'NOMINA')
        os.makedirs(ruta_personalizada, exist_ok=True)

        # --- 3. CÁLCULO DE VALOR HORA INDIVIDUAL ---
        sal_ref_fila = forzar_numero(row.get('SAL_REF', 0))
        v_hora_fila = sal_ref_fila / HR_MES

        datos_emp = {
            'nombre': str(row.get('RAZON_SOCIAL', 'EMPRESA NO ENCONTRADA')),
            'nit': str(row.get('ID_APORTANTE', '000.000.000-0')),
            'tipo': str(row.get('TIPO_DOCUMENTO', 'PERSONA JURÍDICA'))
        }

        pdf = PDF(datos_emp, periodo_actual)
        pdf.add_page()

        # --- BLOQUE INFORMACIÓN EMPLEADO ---
        pdf.set_fill_color(245, 245, 245)
        pdf.set_font('helvetica', 'B', 10)
        pdf.cell(0, 7, f"INFORMACIÓN DEL TRABAJADOR", fill=True, new_x=XPos.LMARGIN, new_y=YPos.NEXT)

        pdf.set_font('helvetica', '', 10)
        pdf.ln(2)
        pdf.cell(95, 6, f"Nombre: {row['NOMBRE_EMPLEADO']}")
        pdf.cell(60, 6, f"Tipo Contrato: {row['TIPO_CONTRATO']}")
        pdf.cell(50, 6, f"Tipo ID: {row['T_ID_EMPLEADO']}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)

        pdf.cell(95, 6, f"Cargo: {row.get('CARGO', 'NO ASIGNADO')}")
        total_dias = forzar_numero(row.get('DIAS_LABORADOS', 0))
        pdf.cell(60, 6, f"Días/Horas Liq: {total_dias:.1f}")
        pdf.cell(50, 6, f"ID: {row['ID_EMPLEADO']}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)

        pdf.ln(5)

        # --- TABLA DE PAGOS ---
        pdf.set_fill_color(230, 230, 230)
        pdf.set_font('helvetica', 'B', 10)
        pdf.cell(100, 8, "DETALLE DE CONCEPTO", border=1, align='C', fill=True)
        pdf.cell(45, 8, "DEVENGADO", border=1, align='C', fill=True)
        pdf.cell(45, 8, "DEDUCIDO", border=1, align='C', fill=True, new_x=XPos.LMARGIN, new_y=YPos.NEXT)

        pdf.set_font('helvetica', '', 10)

        # --- 1. CONCEPTOS DE TIEMPO Y SALARIO (Modificado para añadir Prima) ---
        d_vac = forzar_numero(row.get('DIAS_VACACIONES', 0))
        d_inc = forzar_numero(row.get('DIAS_INCAPACIDAD', 0))
        d_trab = np.maximum(total_dias - d_vac - d_inc, 0)

        label_sueldo = "Sueldo Efectivo" if forzar_numero(row.get('SAL_ESPECIE_PAGADO', 0)) > 0 else "Sueldo por Días Trabajados"

        conceptos_fijos = [
            (f"{label_sueldo} ({d_trab:.0f} días)", 'SUELDO_EFECTIVO_PAGADO'),
            (f"Salario en Especie ({d_trab:.0f} días)", 'SAL_ESPECIE_PAGADO'),
            (f"Vacaciones Disfrutadas ({d_vac:.0f} días)", 'PAGO_VACACIONES'),
            (f"Incapacidad Remunerada ({d_inc:.0f} días)", 'PAGO_INCAPACIDAD'),
            ("Bono No Salarial", 'BONO_PAGADO'),
            ("Auxilio de Transporte", 'VAL_AUX_TTE'),
            ("Prima de Servicios", 'PRIMA_CALC') # <-- Se adiciona concepto de prima al PDF
        ]

        for desc, col in conceptos_fijos:
            val = forzar_numero(row.get(col, 0))
            if val > 0:
                pdf.cell(100, 7, desc, border='LR')
                pdf.cell(45, 7, f"{val:,.0f}", border='LR', align='R')
                pdf.cell(45, 7, "0", border='LR', align='R', new_x=XPos.LMARGIN, new_y=YPos.NEXT)

        # 2. DETALLE DE EXTRAS
        for cod, factor in factores_dict.items():
            cant = forzar_numero(row.get(cod, 0))
            if cant > 0:
                monto = float(cant * v_hora_fila * factor)
                pdf.cell(100, 7, f"{NOMBRES_EXTRAS.get(cod, cod)} ({cant} Hr)", border='LR')
                pdf.cell(45, 7, f"{monto:,.0f}", border='LR', align='R')
                pdf.cell(45, 7, "0", border='LR', align='R', new_x=XPos.LMARGIN, new_y=YPos.NEXT)

        # --- 3. DEDUCCIONES ---
        deducciones = [
            ("Aporte Salud (4%)", 'SALUD_4'),
            ("Aporte Pensión (4%)", 'PENSION_4'),
            ("Descuento por Préstamos", 'PRESTAMOS'),
            ("Salario Especie (Recibido)", 'SAL_ESPECIE_PAGADO')
        ]

        for desc, col in deducciones:
            val = forzar_numero(row.get(col, 0))
            if val > 0:
                pdf.cell(100, 7, desc, border='LR')
                pdf.cell(45, 7, "0", border='LR', align='R')
                pdf.cell(45, 7, f"{val:,.0f}", border='LR', align='R', new_x=XPos.LMARGIN, new_y=YPos.NEXT)

        # LÍNEA FINAL DE LA TABLA
        pdf.cell(190, 0, "", border='T', new_x=XPos.LMARGIN, new_y=YPos.NEXT)

        # TOTALES (Ya reflejarán el aumento en TOTAL_DEVENGADO y NETO_PAGAR que calculó la celda anterior)
        pdf.set_font('helvetica', 'B', 10)
        pdf.cell(100, 8, "TOTALES", border=1, align='R', fill=True)
        pdf.cell(45, 8, f"{forzar_numero(row['TOTAL_DEVENGADO']):,.0f}", border=1, align='R', fill=True)
        pdf.cell(45, 8, f"{forzar_numero(row['TOTAL_DEDUCIDO']):,.0f}", border=1, align='R', fill=True, new_x=XPos.LMARGIN, new_y=YPos.NEXT)

        # NETO A PAGAR
        pdf.ln(4)
        pdf.set_font('helvetica', 'B', 12)
        pdf.cell(145, 10, "NETO A PAGAR:", align='R')
        pdf.set_text_color(0, 50, 150)
        pdf.cell(45, 10, f"${forzar_numero(row['NETO_PAGAR']):,.0f}", border=1, align='C', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_text_color(0, 0, 0)

        # --- FIRMAS ---
        pdf.ln(17)
        y_firma = pdf.get_y()
        pdf.line(25, y_firma, 85, y_firma)
        pdf.line(125, y_firma, 185, y_firma)
        pdf.ln(1)
        pdf.set_font('helvetica', '', 8)
        pdf.cell(95, 3, "Firma del Trabajador", align='C')
        pdf.cell(95, 3, "Firma Autorizada", align='C', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.cell(95, 3, "(Recibí Conforme)", align='C')
        pdf.cell(95, 3, "Empleador / Sello", align='C', new_x=XPos.LMARGIN, new_y=YPos.NEXT)

        # --- NOTAS PIE DE PÁGINA ---
        pdf.ln(10)
        pdf.set_font('helvetica', 'I', 8)
        pdf.set_text_color(100, 100, 100)
        ibc_val = forzar_numero(row.get('IBC_PILA', 0))
        pdf.cell(0, 4, f"* Base de Cotización (IBC): ${ibc_val:,.0f}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.cell(0, 4, f"Estado: {row.get('OBSERVACIONES','LIQUIDADO')} | Generado por Unifika Nómina Cloud.", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.cell(0, 4, "https://unifika.co", new_x=XPos.LMARGIN, new_y=YPos.NEXT, link="https://unifika.co")

        # --- 4. GUARDADO EN AMBAS UBICACIONES ---
        periodo_file = periodo_actual.replace("/", "-").replace(" ", "_")
        nombre_pdf = f"{periodo_file}_{id_contrato}.pdf"

        archivo_personalizado = os.path.join(ruta_personalizada, nombre_pdf)
        pdf.output(archivo_personalizado)

        archivo_general = os.path.join(ruta_drive_comprobantes, nombre_pdf)
        import shutil
        shutil.copy(archivo_personalizado, archivo_general)

    print(f"✅ Proceso finalizado. Comprobantes guardados en: {ruta_drive_comprobantes}.")
    print(f"✅ Se han generado {len(df_activos)} comprobantes en la carpeta /comprobantes")

# Ejecutar
exportar_nomina_multicliente(df_final)

✅ Proceso finalizado. Comprobantes guardados en: /content/drive/MyDrive/Nomina_2026/comprobantes.
✅ Se han generado 274 comprobantes en la carpeta /comprobantes


In [ ]:
import datetime
import pandas as pd
import pytz # Importamos la librería de zonas horarias

def guardar_en_historico_vinculado(df_final_calculado):
    try:
        # 1. Asegurar que existan las columnas de periodo y quincena para la clave
        if 'PERIODO_LIQ' not in df_final_calculado.columns or 'QUINCENA_PAGO' not in df_final_calculado.columns:
            raise ValueError("El DataFrame de entrada debe contener las columnas 'PERIODO_LIQ' y 'QUINCENA_PAGO'")

        # 2. Limpieza de entrada: Solo registros únicos por contrato en esta ejecución
        df_entrada = df_final_calculado.drop_duplicates(subset=['ID_CONTRATO'], keep='last').copy()
        print(f"🔄 Iniciando guardado. Registros únicos a procesar: {len(df_entrada)}")

        # 3. Cargar y limpiar tabla M_APORTANTES
        data_ap = sheet_aportantes.get_all_values()
        df_ap = pd.DataFrame(data_ap[1:], columns=data_ap[0])
        df_ap.columns = [str(c).strip().upper().replace(' ', '_') for c in df_ap.columns]

        # 4. Normalización de IDs
        def limpiar_id(serie):
            return serie.astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

        df_ap['ID_APORTANTE'] = limpiar_id(df_ap['ID_APORTANTE'])
        df_entrada['ID_APORTANTE'] = limpiar_id(df_entrada['ID_APORTANTE'])

        # Eliminar duplicados en la maestra de aportantes
        df_ap_unica = df_ap.drop_duplicates(subset=['ID_APORTANTE'], keep='first')

        # 5. REALIZAR EL CRUCE (Merge)
        cols_a_borrar = [c for c in ['RAZON_SOCIAL', 'TIPO_DOCUMENTO'] if c in df_entrada.columns]
        df_entrada = df_entrada.drop(columns=cols_a_borrar)

        df_h = pd.merge(
            df_entrada,
            df_ap_unica[['ID_APORTANTE', 'RAZON_SOCIAL', 'TIPO_DOCUMENTO']],
            on='ID_APORTANTE',
            how='left'
        )

        # 6. Agregar marca de tiempo con Zona Horaria de Colombia y CLAVE_UNICA
        tz_colombia = pytz.timezone('America/Bogota')
        fecha_proceso = datetime.datetime.now(tz_colombia) # <--- AJUSTE DE HORA COLOMBIA

        df_h['MES_PAGO'] = fecha_proceso.month
        df_h['ANIO_PAGO'] = fecha_proceso.year
        df_h['FECHA_SISTEMA'] = fecha_proceso.strftime("%Y-%m-%d %H:%M:%S")

        # Generamos la Clave Única concatenando los campos solicitados
        df_h['CLAVE_UNICA'] = (
            df_h['ID_CONTRATO'].astype(str) + "_" +
            df_h['PERIODO_LIQ'].astype(str) + "_" +
            df_h['QUINCENA_PAGO'].astype(str)
        )

        # 7. Seleccionar y organizar columnas
        columnas_ordenadas = [
            'CLAVE_UNICA', 'ID_CONTRATO',
            'ANIO_PAGO', 'MES_PAGO', 'TIPO_DOCUMENTO', 'ID_APORTANTE', 'RAZON_SOCIAL',
            'T_ID_EMPLEADO', 'ID_EMPLEADO', 'NOMBRE_EMPLEADO',
            'DIAS_LABORADOS', 'DIAS_VACACIONES', 'DIAS_INCAPACIDAD',
            'SUELDO_PAGADO', 'TOTAL_EXTRAS', 'BONO_PAGADO', 'VAL_AUX_TTE',
            'TOTAL_DEVENGADO', 'PRESTAMOS', 'SALUD_4', 'PENSION_4', 'TOTAL_DEDUCIDO',
            'NETO_PAGAR', 'IBC_PILA', 'FECHA_SISTEMA'
        ]

        df_historico_nuevo = df_h[[c for c in columnas_ordenadas if c in df_h.columns]].copy()

        # 8. --- FORMATEO DE DINERO Y NÚMEROS ---
        cols_moneda = [
            'SUELDO_PAGADO', 'TOTAL_EXTRAS', 'BONO_PAGADO', 'VAL_AUX_TTE',
            'TOTAL_DEVENGADO', 'PRESTAMOS', 'SALUD_4', 'PENSION_4',
            'TOTAL_DEDUCIDO', 'NETO_PAGAR', 'IBC_PILA'
        ]

        for col in cols_moneda:
            if col in df_historico_nuevo.columns:
                df_historico_nuevo[col] = df_historico_nuevo[col].apply(
                    lambda x: f"${int(float(x or 0)):,.0f}".replace(",", ".")
                )

        for col in ['DIAS_LABORADOS', 'DIAS_VACACIONES', 'DIAS_INCAPACIDAD']:
            if col in df_historico_nuevo.columns:
                df_historico_nuevo[col] = df_historico_nuevo[col].apply(
                    lambda x: str(int(float(x or 0)))
                )

        # 9. Conectar a Google Sheets y gestionar duplicados históricos
        try:
            sheet_hist = spreadsheet.worksheet("HIST_NOMINAS")
            # Obtenemos los valores pero limitados a las columnas ordenadas para no leer tus fórmulas
            # columnas_ordenadas tiene 27 columnas (de la 1 a la 27 -> A a la AA)
            num_cols_a_guardar = len(df_historico_nuevo.columns) # Debería ser 27

            # Leemos solo hasta la columna AA para no traer las fórmulas al DataFrame de Python
            data_existente = sheet_hist.get_all_values()

            if len(data_existente) > 1:
                # Cortamos las filas para que solo tengan las columnas del histórico base
                data_limpia = [fila[:num_cols_a_guardar] for fila in data_existente]
                df_existente = pd.DataFrame(data_limpia[1:], columns=data_limpia[0])

                df_total = pd.concat([df_existente, df_historico_nuevo], ignore_index=True)
                df_total = df_total.sort_values(by='FECHA_SISTEMA', ascending=True)
                df_total = df_total.drop_duplicates(subset=['CLAVE_UNICA'], keep='last')

                df_final_guardar = df_total[df_historico_nuevo.columns]
            else:
                df_final_guardar = df_historico_nuevo

        except Exception:
            # Si no existe, crea la hoja con espacio suficiente para las fórmulas (ej. 40 columnas)
            sheet_hist = spreadsheet.add_worksheet(title="HIST_NOMINAS", rows="10000", cols="40")
            df_final_guardar = df_historico_nuevo

        # =====================================================================
        # 10. NUEVA ESTRATEGIA DE ACTUALIZACIÓN PROTEGIENDO FÓRMULAS
        # =====================================================================

        # 1. Determinar el total de filas que vamos a escribir
        total_filas_nuevas = len(df_final_guardar) + 1 # +1 por el encabezado

        # 2. Limpiar SOLAMENTE el bloque de datos antiguo en el rango A2:AA (sin tocar cabeceras ni fórmulas)
        # Esto evita registros fantasmas si la base se achica, pero no borra tus fórmulas de la derecha.
        # Asumiendo 27 columnas (AA), si tienes más o menos, se adaptará dinámicamente usando gspread
        from gspread.utils import rowcol_to_a1

        # Última columna a escribir en formato letra (ej: 'AA')
        letra_ultima_col = rowcol_to_a1(1, num_cols_a_guardar).replace('1', '')

        # Limpiamos por seguridad un rango amplio (ej. hasta la fila 10000) pero SOLO en tus columnas base
        rango_limpieza = f"A2:{letra_ultima_col}10000"
        sheet_hist.batch_clear([rango_limpieza])

        # 3. Preparar los datos exactos (Matriz de strings)
        encabezados = df_final_guardar.columns.values.tolist()
        filas_datos = df_final_guardar.fillna("").astype(str).values.tolist()
        matriz_completa = [encabezados] + filas_datos

        # 4. Definir el rango exacto de escritura (Ej: "A1:AA150")
        rango_escritura = f"A1:{letra_ultima_col}{total_filas_nuevas}"

        # 5. Actualizar la hoja acotando el rango
        sheet_hist.update(range_name=rango_escritura, values=matriz_completa)

        print(f"✅ ¡Éxito! Base de datos actualizada sin afectar columnas formuladas.")
        print(f" Vírgenes y a salvo las columnas desde la {letra_ultima_col} en adelante.")
        return df_final_guardar

    except Exception as e:
        print(f"❌ Error en el guardado histórico: {str(e)}")

# --- Ejecución ---
df_hist_mensual = guardar_en_historico_vinculado(df_final)

🔄 Iniciando guardado. Registros únicos a procesar: 274
✅ ¡Éxito! Base de datos actualizada sin afectar columnas formuladas.
 Vírgenes y a salvo las columnas desde la Y en adelante.
